# Problem Set 1: example solution

Computational Macro, HWS 2026

This notebook generalizes the value function iteration of session 2 to CRRA utility and partial depreciation. The derivations of Part 1 are in the written solution.

### 0. Packages

The first cell activates the course environment in `Julia/` and installs every package listed there
that is missing on your laptop. When nothing is missing it takes a few seconds.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # the course environment in Julia/
Pkg.instantiate()                        # installs every package it lists that is missing on your laptop

In [ ]:
using Plots
using Printf

## Setup

Compared to session 2 three things change (Question 2.1 a): the utility function, the resources $k^\alpha + (1-\delta)k$ and hence the matrices $C$ and $U$, and the steady state that sets the grid bounds. The rest of the loop is unchanged.

In [ ]:
Base.@kwdef struct EconomicParameters
    α::Float64 = 0.3
    β::Float64 = 0.96
    γ::Float64 = 1.0
    δ::Float64 = 1.0
end

Base.@kwdef struct NumericalParameters
    nk::Int = 200
    kmin_rel::Float64 = 0.1
    kmax_rel::Float64 = 2.0
    crit::Float64 = 1e-6
    maxiter::Int = 5000
end

util(c, γ) = γ == 1.0 ? log(c) : (c^(1 - γ) - 1) / (1 - γ)

resources(k, par) = k .^ par.α .+ (1 - par.δ) .* k

steady_state(par) = (par.α / (1 / par.β - 1 + par.δ))^(1 / (1 - par.α))

In [ ]:
function solve_vfi(par, mpar)
    println("Value function iteration with parameters alpha = $(par.α), beta = $(par.β), ",
            "gamma = $(par.γ), delta = $(par.δ), N = $(mpar.nk), crit = ", @sprintf("%g", mpar.crit))
    kstar = steady_state(par)
    k = collect(range(mpar.kmin_rel * kstar, mpar.kmax_rel * kstar, length = mpar.nk))

    C = resources(k, par) .- k'                   # rows: k_i, columns: k'_j
    U = fill(-Inf, size(C))
    U[C .> 0] = util.(C[C .> 0], par.γ)

    v = zeros(mpar.nk)
    g = ones(Int, mpar.nk)
    dist, it = Inf, 0
    t0 = time()
    while dist > mpar.crit && it < mpar.maxiter
        M = U .+ par.β .* v'
        vals, idx = findmax(M, dims = 2)
        vnew, g = vec(vals), vec(getindex.(idx, 2))
        dist = maximum(abs.(vnew .- v))
        v = vnew
        it += 1
    end
    runtime = time() - t0
    dist > mpar.crit && error("VFI did not converge")

    @printf("  iterations: %d, final distance: %.2e, policy at lower bound: %d, at upper bound: %d\n",
            it, dist, count(g .== 1), count(g .== mpar.nk))
    c = resources(k, par) .- k[g]
    return (k = k, v = v, g = g, c = c, it = it, time = runtime, kstar = kstar)
end

function euler_error(c, kprime, cnext, par)
    R = par.α .* kprime .^ (par.α - 1) .+ (1 - par.δ)
    return 1 .- (par.β .* R) .^ (-1 / par.γ) .* cnext ./ c
end

# tomorrow's capital k_{g_i} is on the grid, so tomorrow's consumption is c_{g_i}
euler_error_grid(sol, par) = euler_error(sol.c, sol.k[sol.g], sol.c[sol.g], par)

function simulate(sol, k0; T = 300)
    idx = zeros(Int, T + 1)
    idx[1] = argmin(abs.(sol.k .- k0))
    for t in 1:T
        idx[t + 1] = sol.g[idx[t]]
    end
    return sol.k[idx]
end

## Question 2.1 (b)–(d): log utility and full depreciation

In [ ]:
par = EconomicParameters()
mpar = NumericalParameters()
sol = solve_vfi(par, mpar)

# closed form
k = sol.k
F = par.α / (1 - par.α * par.β)
E = (log(1 - par.α * par.β) + par.β * F * log(par.α * par.β)) / (1 - par.β)
kp_true = par.α * par.β .* k .^ par.α
@printf("(b) policy error: %.2e, value error: %.2e\n",
        maximum(abs.(k[sol.g] .- kp_true)), maximum(abs.(sol.v .- (E .+ F .* log.(k)))))

# (c) test with the closed form policy
c_true = (1 - par.α * par.β) .* k .^ par.α
cnext_true = (1 - par.α * par.β) .* kp_true .^ par.α
@printf("(c) max |e| with closed form: %.1e\n", maximum(abs.(euler_error(c_true, kp_true, cnext_true, par))))

# (d) on-grid policy
e = euler_error_grid(sol, par)
i = argmax(abs.(e))
@printf("(d) max |e| = %.2e at grid point %d, k = %.4f\n", abs(e[i]), i, k[i])

We reproduce the numbers from class (337 iterations, policy error $9.4 \cdot 10^{-4}$, value error $2.4 \cdot 10^{-5}$), and the Euler equation error of the closed form is zero up to floating point precision. For the grid solution, consumption at the second grid point $k = 0.0185$ deviates by 1.2 percent from what the Euler equation prescribes.

## Question 2.1 (e)–(f): $\gamma = 2$, $\delta = 0.1$

In [ ]:
par2 = EconomicParameters(γ = 2.0, δ = 0.1)
sol2 = solve_vfi(par2, mpar)
k2, g2 = sol2.k, sol2.g
e2 = euler_error_grid(sol2, par2)
i = argmax(abs.(e2))
@printf("(e) max |e| = %.2e at grid point %d, k = %.4f, mean log10|e| = %.2f\n",
        abs(e2[i]), i, k2[i], sum(log10.(abs.(e2))) / length(e2))

fixed = k2[g2 .== 1:mpar.nk]
@printf("(f) k* = %.4f, ", sol2.kstar)
println("grid points with k' = k: ", round.(fixed, digits = 4))

p1 = plot(k2, sol2.v, title = "Value function", label = "")
p2 = plot(k2, k2[g2], title = "Policy function", label = "k'", legend = :topleft)
plot!(p2, k2, k2, color = :black, linestyle = :dash, linewidth = 0.8, label = "45 degree line")
p3 = plot(k2, sol2.c, title = "Consumption", label = "")
plot(p1, p2, p3, layout = (1, 3), size = (1400, 400), xlabel = "k", margin = 5Plots.mm)

The largest Euler equation error is 4.6 percent of consumption, at the first grid point $k = 0.292$, and the mean of $\log_{10}|e_i|$ is $-2.16$. The policy crosses the 45 degree line in the three grid points around $k^* = 2.921$, so the two agree up to the grid spacing. The error is largest at low $k$, where consumption is smallest: the policy misses the exact choice by up to one grid spacing, and this error is measured relative to $c$.

## Question 2.2 (a): simulation

In [ ]:
path = simulate(sol2, 0.5 * sol2.kstar)
@printf("path settles at %.4f, k* = %.4f, difference %.1f percent\n",
        path[end], sol2.kstar, 100 * (path[end] / sol2.kstar - 1))

plot(0:300, path, label = "k_t", xlabel = "t", legend = :bottomright)
hline!([sol2.kstar], color = :black, linestyle = :dash, linewidth = 0.8, label = "k*")

The path settles at 2.886, 1.2 percent below $k^*$. It stops at the first grid point with $g_i = i$ it reaches from below.

## Question 2.2 (b)–(c): the cost of the grid

In [ ]:
rows = []
for nk in [50, 200, 800]
    s = solve_vfi(par2, NumericalParameters(nk = nk))
    push!(rows, (nk, s.k[2] - s.k[1], s.it, s.time,
                 maximum(abs.(euler_error_grid(s, par2))), simulate(s, 0.5 * s.kstar)[end]))
end

@printf("\n%5s %8s %5s %7s %9s %8s\n", "N", "spacing", "iter", "time", "max|e|", "settles")
for r in rows
    @printf("%5d %8.4f %5d %7.3f %9.2e %8.4f\n", r[1], r[2], r[3], r[4], r[5], r[6])
end

N_needed = 800 * rows[end][5] / 1e-4
@printf("\n(c) N for max|e| < 1e-4: %.0f, time: %.0f minutes\n", N_needed, rows[end][4] * (N_needed / 800)^2 / 60)

The number of iterations does not depend on $N$, but each iteration works on an $N \times N$ matrix, so the run time is of order $N^2$ (factor of about 16 from 200 to 800). The maximum Euler error is proportional to the grid spacing and falls like $1/N$ (factor of about 4). To get it below $10^{-4}$ we would need about 89,000 grid points, which would take around an hour and 63 GB for a single matrix.

## Question 2.2 (d)–(e): risk aversion

In [ ]:
mpar800 = NumericalParameters(nk = 800)
kstar = steady_state(EconomicParameters(δ = 0.1))     # the same for all γ
p = plot(xlabel = "t", legend = :bottomright)
for gam in [1.0, 2.0, 5.0]
    par_g = EconomicParameters(γ = gam, δ = 0.1)
    s = solve_vfi(par_g, mpar800)
    path_g = simulate(s, 0.5 * s.kstar)
    t5 = findfirst(abs.(path_g .- s.kstar) ./ s.kstar .< 0.05) - 1
    i0 = argmin(abs.(s.k .- 0.5 * s.kstar))
    k0 = s.k[i0]
    srate = (s.k[s.g[i0]] - (1 - par_g.δ) * k0) / k0^par_g.α
    @printf("  periods to within 5%% of k*: %d, saving rate at k0: %.3f\n", t5, srate)
    plot!(p, 0:99, path_g[1:100], label = "γ = $(Int(gam))")
end
hline!(p, [kstar], color = :black, linestyle = :dash, linewidth = 0.8, label = "k*")

The household with $\gamma = 1$ reaches the steady state fastest (15 periods, against 22 for $\gamma = 2$ and 37 for $\gamma = 5$). At $k_0$ the return on capital is above $1/\beta$, so the household wants consumption to grow. With a high intertemporal elasticity of substitution $1/\gamma$ it accepts a steep consumption path and saves more (saving rate 0.316 against 0.198 for $\gamma = 5$), so capital accumulates faster. The steady state does not depend on $\gamma$.